# 参考题解：QLoRA 量化低秩线性层

实现不依赖 bitsandbytes、可在 CPU 评测的 QLoRA 核心机制。

核心思路：把冻结的基础权重对称量化到 4-bit 范围 `[-8, 7]`，前向时反量化，再叠加可训练的 LoRA 低秩更新。


In [ ]:
# ✅ SOLUTION

import torch
import torch.nn as nn
import torch.nn.functional as F

class QLoRALinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, rank: int = 4, alpha: float = 1.0):
        super().__init__()
        self.rank, self.alpha = rank, alpha
        weight = torch.empty(out_features, in_features)
        nn.init.kaiming_uniform_(weight, a=5 ** 0.5)
        scale = weight.abs().amax(dim=1, keepdim=True).clamp_min(1e-8) / 7.0
        qweight = torch.clamp(torch.round(weight / scale), -8, 7).to(torch.int8)
        self.register_buffer("qweight", qweight)
        self.register_buffer("weight_scale", scale)
        self.A = nn.Parameter(torch.empty(rank, in_features))
        self.B = nn.Parameter(torch.zeros(out_features, rank))
        nn.init.kaiming_uniform_(self.A, a=5 ** 0.5)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base = F.linear(x, self.qweight.float() * self.weight_scale)
        update = F.linear(F.linear(x, self.A), self.B) * (self.alpha / self.rank)
        return base + update
